# 05 · Information rather than representativeness

Does information-based selection improve the **same 1D 2PL** baseline?

| Selector | Estimators |
|---|---|
| random | accuracy, p-IRT, gp-IRT |
| IRT anchors | accuracy, p-IRT, gp-IRT |
| fisher_static | accuracy, p-IRT, gp-IRT |
| fisher_adaptive | accuracy, p-IRT, gp-IRT |

Use the baseline's five family splits, 20 seeds and k = 10, 20, 50, 100, 200, 500.
Fit item parameters on train; fit the existing gp-IRT blend **per selector and k on validation**;
then evaluate test. Accuracy from deliberately informative items is generally biased, so keep
it as an ablation. The validation-fitted blend can learn to give it little weight.

Our parameters are $p_i=\sigma(\alpha_i\theta-\beta_i)$, so
$I_i(\theta)=\alpha_i^2 p_i(1-p_i)$. Static Fisher averages this over training abilities.
Adaptive Fisher starts at the training mean, reveals one answer, refits ability with frozen items,
and repeats. One trajectory supplies every k checkpoint; it is shared across seeds.

The original baseline keeps Adam inference. These experiments use an equivalent, selected-answer-only
Newton MAP fit for speed. No item model or prior is tuned on test.

In [ ]:
import sys

sys.path.insert(0, "../scripts/irt")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from efficbench import ROOT, load
from protocol import fit_context, make_splits
from run_progression import INFORMATION_SELECTORS, run_selection

load_data = load  # Swap the loader while keeping response / model / item column order aligned.
ks = [10, 20, 50, 100, 200, 500]
seeds = range(20)

Y, models, items = load_data()
Y = Y.astype(np.float32)
splits = make_splits(Y)
subjects, model_ids = items.subject.values, models.model_id.values
item_ids = items.item_id.values


out = ROOT / "results/irt/progression/information"

# Replace this factory, or supply one frozen adapter per split below.
build_context = fit_context
contexts = None

## Model plug-in point

These stages do not require our fitter. Set `contexts` (or `contexts_by_dim` in 06) to
**one training-only frozen model per family split**, in that split order. Or replace
`build_context(Y_train, subjects, n_dims=...)` with your teammate's fitter.

For fitted 1PL/2PL/MIRT parameters:

```python
from cabench.design import parameter_context
ctx = parameter_context(train_theta, item_alpha, item_beta, item_ids=item_ids)
# Optional: retain your teammate's ability estimator (responses contains ONLY selected answers).
ctx["infer"] = teammate_infer  # (responses, items, initial=None) -> (models, D)
```

`item_beta` is the intercept in `sigmoid(theta @ alpha.T - beta)`; if the fit uses
`sigmoid(a * (theta-b))`, pass `beta=a*b`. Use ones for Rasch discrimination.
For a different backend, use `model_context` with prediction / inference callbacks and fixed
loadings; non-logistic links also need their correct `information_weights` callback.
See `scripts/irt/README.md` for shapes and examples. Keep item IDs, latent coordinates,
training prior and response columns aligned. Use a distinct `out` directory per fitted model.
The experiment helpers perform no refit when supplied with contexts.

## Fit, calibrate, evaluate

This runs all 12 combinations. It writes aggregate metrics, per-model predictions,
validation predictions, gp weights and paired comparisons. `baseline.csv` is untouched.

In [ ]:
if contexts is None:
    contexts = [build_context(Y[train], subjects, n_dims=1) for train, _, _ in splits]
metrics, records = run_selection(
    Y,
    splits,
    contexts,
    INFORMATION_SELECTORS,
    out,
    model_ids,
    ks=ks,
    seeds=seeds,
    item_ids=item_ids,
)

## Selection with the estimator held fixed

Mean test MAE (percentage points), averaging seeds within a split and then the five splits.
Compare gp-IRT across selectors; p-IRT versus gp-IRT below isolates the estimator change.

In [ ]:
gp = metrics[metrics.estimator == "gp-IRT"]
per_split = gp.groupby(["selector", "k", "split"]).mae.mean()
summary = per_split.groupby(["selector", "k"]).agg(["mean", "std"])

fig, ax = plt.subplots(figsize=(8, 4.5))
for selector in INFORMATION_SELECTORS:
    curve = summary.loc[selector]
    ax.errorbar(
        curve.index,
        100 * curve["mean"],
        yerr=100 * curve["std"],
        marker="o",
        capsize=3,
        label=selector,
    )
ax.set(
    xscale="log",
    xlabel="k (answers observed)",
    ylabel="test MAE (pp)",
    title="1D 2PL + gp-IRT (±1 SD across splits)",
)
ax.legend()
plt.tight_layout()

In [ ]:
100 * metrics.groupby(["k", "selector", "estimator"]).mae.mean().unstack("estimator")

## Paired changes against random + gp-IRT

Negative ΔMAE means improvement. The percentile bootstrap resamples paired **split means**,
after averaging seeds within models. Repeated deterministic trajectories do not count as independent
trials. These five family splits overlap: the CI describes sensitivity to these splits,
not five independent populations.

In [ ]:
paired = pd.read_csv(out / "paired.csv")
headline = paired[paired.method.str.endswith("+ gp-IRT")].copy()
headline[["delta_mae", "ci_low", "ci_high"]] *= 100
headline[["method", "k", "delta_mae", "ci_low", "ci_high", "splits_won", "n_splits"]]

In [ ]:
pd.read_csv(out / "gp_weights.csv").groupby(["selector", "k"]).gp_weight.agg(["mean", "min", "max"])

this follow Fisher-Information-based adaptation but it stops at 1D information selection. Dimensionality is a separate experiment in 06.